In [ ]:
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import os
import glob 

# ---- Paths - EDIT THESE ----
BIDS_DIR    = "/path/to/bids"
RESULTS_DIR = "/path/to/results"                         # same results folder as the processing scripts
REG_DIR     = os.path.join(RESULTS_DIR, "04_registration")  # registration_pipe.sh output
MODEL_DIR   = os.path.join(RESULTS_DIR, "05_models")        # model summary CSVs
PLOT_DIR    = os.path.join(RESULTS_DIR, "06_figures")       # figures

SUB_DF = pd.read_csv(os.path.join(BIDS_DIR, "participants.tsv"), sep='\t')

controls = SUB_DF.loc[SUB_DF['group_id'] == 0, 'participant_id'].to_list()
cons = SUB_DF.loc[SUB_DF['group_id'] == 1, 'participant_id'].to_list()
subs = SUB_DF['participant_id'].to_list()

SUB_DF

,participant_id,group_id,age,sex,rpq-13
0,sub-01,0,29,M,NaN
1,sub-02,0,27,F,NaN
2,sub-03,0,24,M,NaN
3,sub-04,0,28,M,NaN
4,sub-05,0,23,M,NaN
...,...,...,...,...,...
57,sub-67,1,22,F,14.0
58,sub-68,1,29,F,7.0
59,sub-69,1,21,M,32.0
60,sub-70,1,20,M,NaN


In [ ]:
import numpy as np

# Parameters to extract
DTD_PARAMS = ['ufa', 'msqanison', 'op', 'miso', 'vison']
DTI_PARAMS = ['FA', 'MD']

#make this directory and add roi masks to tract categories based on Fig. 1 in Rizor et al. or your own definitions 
ROI_DIR    = os.path.join(REG_DIR, "ROIs_CONspace", "final_regions")
assoc_rois = sorted(glob.glob(f"{ROI_DIR}/Association/MNI*.nii.gz"))
comm_rois  = sorted(glob.glob(f"{ROI_DIR}/Commissural/MNI*.nii.gz"))
proj_rois  = sorted(glob.glob(f"{ROI_DIR}/Projection/MNI*.nii.gz"))

assoc_names = ['AFasc_L', 'AFasc_R', 'Cing_L', 'Cing_R', 'IFOF_L', 'IFOF_R',
               'ILF_L', 'ILF_R', 'SLF1_L', 'SLF1_R', 'SLF2_L', 'SLF2_R',
               'SLF3_L', 'SLF3_R', 'UF_L', 'UF_R']
comm_names  = ['CC_Body', 'CC_FMaj', 'CC_FMin', 'CC_Tap']
proj_names  = ['CBT_L', 'CBT_R', 'CPT_L', 'CPT_R', 'CST_L', 'CST_R',
               'CSTr_L', 'CSTr_R', 'RS_L', 'RS_R']

rois       = assoc_rois + comm_rois + proj_rois
mask_names = assoc_names + comm_names + proj_names

roi_type_lookup = (
    {name: 'association' for name in assoc_names} |
    {name: 'commissural' for name in comm_names}  |
    {name: 'projection'  for name in proj_names}
)

# pre-load full ROI masks
roi_masks = {}
for i, roi in enumerate(rois):
    roi_masks[mask_names[i]] = nib.load(roi).get_fdata().astype(bool)

# ------------------------------------------------------------------ #
records = []

# --- define transforms ---
TRANSFORMS = {
    'log':    np.log,
    'reverse_log': lambda x: np.where((x > 0) & (x < 1), -np.log(1 - x), np.nan),
    'scale_log': lambda x: np.log(x * 1e9),
    'sqrt':   np.sqrt,
    'square': np.square,
    'abs':    np.abs,
    None:     lambda x: x   # no transform
}

def apply_transform(voxels, transform=None):
    """Apply transform to voxel values before computing median."""
    if transform is None:
        return voxels
    fn = TRANSFORMS.get(transform)
    if fn is None:
        raise ValueError(f"Unknown transform '{transform}'. "
                         f"Choose from {list(TRANSFORMS.keys())}")
    transformed = fn(voxels)
    # remove any non-finite values introduced by transform (e.g. log of negative)
    return transformed[np.isfinite(transformed)]


# --- updated extraction loop ---
PARAM_TRANSFORM = {
    'ufa':       None,
    'msqanison': None,
    'op':        None,
    'miso':      None,
    'vison':     'log', #vison log transformed due to skew 
    'FA':        None,
    'MD':        None,
}  # set per-param transforms here e.g. 'MD': 'log'

records = []

for subject in SUB_DF['participant_id']:
    wm_path = os.path.join(REG_DIR, subject, f"{subject}_WM_mask_bin.nii.gz")
    wm_mask = nib.load(wm_path).get_fdata().astype(bool)

    for param, is_dtd in [(p, True) for p in DTD_PARAMS] + [(p, False) for p in DTI_PARAMS]:

        if is_dtd:
            param_path = os.path.join(REG_DIR, subject, "dtd_CONspace",
                f"{subject}_dtd_{param}_CONspace.nii.gz")
        else:
            param_path = os.path.join(REG_DIR, subject, "dti_CONspace",
                f"{subject}_{param}_CONspace.nii.gz")

        if not os.path.exists(param_path):
            print(f"WARNING: missing {param} for {subject}")
            continue

        param_img = nib.load(param_path).get_fdata(dtype=np.float64)
        transform = PARAM_TRANSFORM.get(param, None)

        # --- full ROI medians ---
        for roi_name, mask in roi_masks.items():
            comb_mask = wm_mask & mask
            voxels    = param_img[comb_mask]
            voxels    = voxels[np.isfinite(voxels)]
            voxels    = apply_transform(voxels, transform)

            records.append({
                'participant_id': subject,
                'parameter':      param,
                'param_type':     'dtd' if is_dtd else 'dti',
                'roi':            roi_name,
                'tract_type':     roi_type_lookup[roi_name],
                'median_val':     np.median(voxels) if len(voxels) > 0 else np.nan,
                'n_voxels':       len(voxels),
                'transform':      transform or 'none',
            })

median_df = pd.DataFrame(records)
long_df   = median_df.merge(SUB_DF, on='participant_id')
#drop_ids = ['sub-03', 'sub-08', 'sub-17', 'sub-29', 'sub-33', 'sub-38'] # optional, drop controls who had previus concussion history 
#long_df = long_df[~long_df['participant_id'].isin(drop_ids)]

/var/folders/ph/gfnzgrf17f55k76wdlf728rc0000gn/T/ipykernel_52483/3602284649.py:74: RuntimeWarning: divide by zero encountered in log
  transformed = fn(voxels)


In [ ]:
# prepare long_df for pymc modeling
long_df['roi_idx']     = pd.Categorical(long_df['roi']).codes
long_df['subj_idx']    = pd.Categorical(long_df['participant_id']).codes
long_df['sex_numeric'] = long_df['sex'].map({'M': 0, 'F': 1})

# tract type index
long_df['tract_type_idx'] = pd.Categorical(long_df['tract_type']).codes

# label mappings
roi_labels        = dict(enumerate(pd.Categorical(long_df['roi']).categories))
subj_labels       = dict(enumerate(pd.Categorical(long_df['participant_id']).categories))
tract_type_labels = dict(enumerate(pd.Categorical(long_df['tract_type']).categories))

# roi -> tract type index mapping (one entry per ROI)
roi_to_tract_type = (
    long_df[['roi_idx', 'tract_type_idx']]
    .drop_duplicates()
    .sort_values('roi_idx')
    ['tract_type_idx']
    .values
)

print(roi_labels)
print(tract_type_labels)
print(f"ROI → tract type mapping: {roi_to_tract_type}")
print(f"N ROIs:        {long_df['roi_idx'].nunique()}")
print(f"N tract types: {long_df['tract_type_idx'].nunique()}")
print(f"N subjects:    {long_df['subj_idx'].nunique()}")

from scipy.stats import zscore

# z-score within parameter × roi
long_df['median_zscore'] = long_df.groupby(
    ['parameter', 'roi']
)['median_val'].transform(
    lambda x: zscore(x, ddof=1)
)

# verify
print(long_df.groupby(['parameter', 'roi'])['median_zscore']
      .agg(['mean', 'std']).round(3))

In [ ]:
import pymc as pm

def run_hierarchical_logistic(long_df, param_name, fiber_type):

    df = long_df[
        (long_df['parameter']  == param_name)  &
        (long_df['tract_type'] == fiber_type)
    ].copy()

    if df.empty:
        print(f"  No data for {param_name} / {fiber_type}, skipping")
        return None

    df['roi_idx_local'] = pd.Categorical(df['roi']).codes
    n_rois              = df['roi_idx_local'].nunique()
    roi_labels          = dict(enumerate(pd.Categorical(df['roi']).categories))

    y       = df['group_id'].values.astype(int)
    metric  = df['median_zscore'].values.astype(float)
    age_z   = ((df['age'] - df['age'].mean()) / df['age'].std()).values.astype(float)
    sex     = df['sex_numeric'].values.astype(float)
    roi_idx = df['roi_idx_local'].values

    print(f"  {param_name} | {fiber_type}: {n_rois} ROIs, {len(y)} obs")

    with pm.Model() as logistic_model:

        # --- Level 2: fiber type hyperprior ---
        mu_beta    = pm.Normal('mu_beta',    mu=0, sigma=1)
        sigma_beta = pm.HalfNormal('sigma_beta', sigma=0.5)

        # --- Level 1: ROI effects pooled toward fiber type mean ---
        roi_offset  = pm.Normal('roi_offset', mu=0, sigma=1, shape=n_rois)
        beta_metric = pm.Deterministic('beta_metric',
                                        mu_beta + roi_offset * sigma_beta)

        # --- Fixed effects ---
        beta_age  = pm.Normal('beta_age',  mu=0, sigma=1)
        beta_sex  = pm.Normal('beta_sex',  mu=0, sigma=1)
        intercept = pm.Normal('intercept', mu=0, sigma=1)

        # --- Linear predictor ---
        logit_p = (intercept +
                   beta_metric[roi_idx] * metric +
                   beta_age  * age_z +
                   beta_sex  * sex)

        pm.Bernoulli('y', logit_p=logit_p, observed=y)

        idata = pm.sample(
            5000, tune=5000,
            target_accept=0.95,
            return_inferencedata=True,
            random_seed=8,
            progressbar=True,
        )

        idata.attrs['param']      = param_name
        idata.attrs['fiber_type'] = fiber_type
        idata.attrs['roi_labels'] = str(roi_labels)

    return idata


# --- run for all params x fiber types ---
fiber_types  = long_df['tract_type'].unique()

# nested dict: logistic_results[param][fiber_type]
logistic_results = {}

for param in DTD_PARAMS + DTI_PARAMS:
    logistic_results[param] = {}
    for fiber_type in fiber_types:
        print(f"\nFitting: {param} | {fiber_type}...")
        idata = run_hierarchical_logistic(long_df, param, fiber_type)
        if idata is not None:
            logistic_results[param][fiber_type] = idata
            print(f"  ✓ {param} | {fiber_type} complete")

In [ ]:
import arviz as az

for param, fiber_dict in logistic_results.items():
    for fiber_type, idata in fiber_dict.items():
        if idata is None:
            print(f"  {param} | {fiber_type}: no results, skipping")
            continue

        diag = az.summary(
            idata,
            var_names=['mu_beta', 'beta_metric'],
            hdi_prob=0.95
        )[['r_hat', 'ess_bulk']]

        bad_rhat = diag[diag['r_hat'] > 1.01]
        low_ess  = diag[diag['ess_bulk'] < 400]

        if len(bad_rhat) > 0 or len(low_ess) > 0:
            print(f"WARNING {param} | {fiber_type}:")
            if len(bad_rhat) > 0:
                print(f"  High r_hat:  {bad_rhat.index.tolist()}")
            if len(low_ess) > 0:
                print(f"  Low ESS:     {low_ess.index.tolist()}")
        else:
            n_div = idata.sample_stats['diverging'].values.sum()
            print(f"  ✓ {param} | {fiber_type}: converged OK  (divergences: {n_div})")

In [ ]:
import pandas as pd
import numpy as np
import arviz as az
import ast

def extract_idata_to_rows(idata, param, fiber_type):
    """Extract posterior summaries from a single idata object into a list of rows."""
    rows = []
    
    # Recover roi_labels from attrs
    roi_labels = ast.literal_eval(idata.attrs.get('roi_labels', '{}'))
    
    # Variables to summarize
    scalar_vars = ['mu_beta', 'sigma_beta', 'beta_age', 'beta_sex', 'intercept']
    vector_vars = ['beta_metric', 'roi_offset']  # shape = (n_rois,)
    
    summary = az.summary(idata, var_names=scalar_vars + vector_vars, hdi_prob=0.95)
    
    base = dict(param=param, fiber_type=fiber_type)
    
    # --- Scalar variables ---
    for var in scalar_vars:
        if var not in summary.index:
            continue
        row = summary.loc[var]
        rows.append({
            **base,
            'variable':  var,
            'roi_index': None,
            'roi_label': None,
            'mean':      row['mean'],
            'sd':        row['sd'],
            'hdi_2.5%':  row['hdi_2.5%'],
            'hdi_97.5%': row['hdi_97.5%'],
            'r_hat':     row.get('r_hat', np.nan),
            'ess_bulk':  row.get('ess_bulk', np.nan),
        })
    
    # --- Vector variables (one row per ROI) ---
    for var in vector_vars:
        for i, label in roi_labels.items():
            idx_str = f'{var}[{i}]'
            if idx_str not in summary.index:
                continue
            row = summary.loc[idx_str]
            rows.append({
                **base,
                'variable':  var,
                'roi_index': i,
                'roi_label': label,
                'mean':      row['mean'],
                'sd':        row['sd'],
                'hdi_2.5%':  row['hdi_2.5%'],
                'hdi_97.5%': row['hdi_97.5%'],
                'r_hat':     row.get('r_hat', np.nan),
                'ess_bulk':  row.get('ess_bulk', np.nan),
            })
    
    return rows


def results_to_csvs(logistic_results, out_dir='.'):
    """
    Writes three CSVs from logistic_results:
      1. summary_all.csv          — all variables, all models
      2. summary_fixed_effects.csv — scalar fixed effects only
      3. summary_roi_effects.csv   — ROI-level beta_metric rows only
    """
    import os
    os.makedirs(out_dir, exist_ok=True)
    
    all_rows = []
    
    for param, fiber_dict in logistic_results.items():
        for fiber_type, idata in fiber_dict.items():
            print(f"Extracting: {param} | {fiber_type}")
            try:
                rows = extract_idata_to_rows(idata, param, fiber_type)
                all_rows.extend(rows)
            except Exception as e:
                print(f"  ✗ Failed: {e}")
    
    df_all = pd.DataFrame(all_rows)
    
    # Add significance flag: HDI doesn't include 0
    df_all['hdi_excludes_zero'] = (
        (df_all['hdi_2.5%'] > 0) | (df_all['hdi_97.5%'] < 0)
    )
    
    # --- Write CSVs ---
    path_all = os.path.join(out_dir, 'summary_all.csv')
    df_all.to_csv(path_all, index=False)
    print(f"\nWrote: {path_all}  ({len(df_all)} rows)")
    
    fixed_vars = ['mu_beta', 'sigma_beta', 'beta_age', 'beta_sex', 'intercept']
    df_fixed = df_all[df_all['variable'].isin(fixed_vars)]
    path_fixed = os.path.join(out_dir, 'summary_fixed_effects.csv')
    df_fixed.to_csv(path_fixed, index=False)
    print(f"Wrote: {path_fixed}  ({len(df_fixed)} rows)")
    
    df_roi = df_all[df_all['variable'] == 'beta_metric']
    path_roi = os.path.join(out_dir, 'summary_roi_effects.csv')
    df_roi.to_csv(path_roi, index=False)
    print(f"Wrote: {path_roi}  ({len(df_roi)} rows)")
    
    return df_all


# --- Run it ---
df_results = results_to_csvs(logistic_results, out_dir=MODEL_DIR)

In [ ]:
PARAM_COLORS = {
    'ufa':   '#E63946',
    'FA':    '#F4A261',
    'op':    '#FF6B35',
    'vison': '#2A9D8F',
    'miso':  '#457B9D',
    'MD':    '#6A4C93',
}

ANISOTROPIC_PARAMS = ['ufa', 'FA', 'op']
ISOTROPIC_PARAMS  = ['vison', 'MD']

def plot_mu_betas_grouped(logistic_results, param_group, group_name,
                          save_dir=None):
    FIBER_ORDER = ['association', 'commissural', 'projection']

    available_params = [
        p for p in param_group
        if p != 'msqanison' and p in logistic_results
    ]

    n_fibers     = len(FIBER_ORDER)
    n_params     = len(available_params)
    group_width  = 0.8
    violin_width = group_width / n_params

    fig, ax = plt.subplots(figsize=(n_fibers * 2.5 + 1, 5))
    handles = []

    for p_idx, param in enumerate(available_params):
        color        = PARAM_COLORS.get(param, '#888888')
        positions    = []
        samples_list = []
        medians      = []
        hdis         = []

        for f_idx, fiber_type in enumerate(FIBER_ORDER):
            idata = (logistic_results
                     .get(param, {})
                     .get(fiber_type))
            if idata is None:
                continue

            samples = idata.posterior['mu_beta'].values.ravel()
            pos     = f_idx + (p_idx - n_params / 2 + 0.5) * violin_width

            positions.append(pos)
            samples_list.append(samples)
            medians.append(np.median(samples))
            hdis.append(az.hdi(samples, hdi_prob=0.95))

        if not samples_list:
            continue

        parts = ax.violinplot(
            samples_list, positions=positions,
            widths=violin_width * 0.9,
            vert=True, showmedians=False, showextrema=False,
        )
        for pc, hdi in zip(parts['bodies'], hdis):
            contains_zero = hdi[0] < 0 < hdi[1]
            pc.set_facecolor(color)
            pc.set_alpha(0.1 if contains_zero else 0.6)
            pc.set_edgecolor(color)
            pc.set_linewidth(0.8)

        for pos, med, hdi, samp in zip(positions, medians, hdis, samples_list):
            contains_zero = hdi[0] < 0 < hdi[1]
            alpha = 0.1 if contains_zero else 0.6
            ax.plot([pos, pos], [hdi[0], hdi[1]],
                    color='black', linewidth=2.5, alpha=alpha,
                    solid_capstyle='round', zorder=3)
            ax.scatter(pos, med, facecolors='black', edgecolors='black', alpha=alpha, s=20, zorder=4)
            if hdi[0] > 0 or hdi[1] < 0:
                y_star = np.percentile(samp, 100)
                ax.text(pos, y_star, '*', ha='center', va='bottom',
                        fontsize=20, color='black', zorder=5)

        handles.append(plt.Line2D([0], [0], color=color, linewidth=6,
                                   alpha=0.6, label=param))

    ax.axhline(0, color='gray', linewidth=1.2, linestyle='--', zorder=1)
    ax.set_xticks(np.arange(n_fibers))
    ax.set_xticklabels([f.capitalize() for f in FIBER_ORDER], fontsize=18)
    ax.tick_params(axis='y', labelsize=18)
    ax.set_ylabel("Log-odds of Concussion", fontsize=24)
    #ax.set_title(f"{region_label}", fontsize=20)
    ax.spines[['top', 'right']].set_visible(False)
    
    if group_name == "Anisotropic":
        ax.legend(handles=handles, frameon=False, loc = 'upper left', fontsize=14, labels=[r'$\mathrm{\mu}$FA', "FA", "OP"])
    elif group_name == "Isotropic":
        ax.legend(handles=handles, frameon=False, loc = 'upper left', fontsize=14, labels=[r'$\hat{V}(D_{\mathrm{iso}})$', "MD"])

    plt.tight_layout()

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        tag   = group_name.lower().replace(' ', '_')
        fname = f"{save_dir}/mu_beta_{tag}.png"
        plt.savefig(fname, dpi=600, bbox_inches='tight')
        print(f"  ✓ saved: {fname}")
    else:
        plt.show()
    plt.close()


# --- call for each parameter group ---
plot_mu_betas_grouped(logistic_results, ANISOTROPIC_PARAMS,
                      "Anisotropic", save_dir=PLOT_DIR)
plot_mu_betas_grouped(logistic_results, ISOTROPIC_PARAMS,
                      "Isotropic",  save_dir=PLOT_DIR)

In [ ]:
import pymc as pm

def run_op_regression(long_df, fiber_type):

    df = long_df[
        (long_df['tract_type'] == fiber_type)
    ].copy()

    if df.empty:
        print(f"  No data for {fiber_type}, skipping")
        return None

    df['roi_idx_local'] = pd.Categorical(df['roi']).codes
    n_rois              = df['roi_idx_local'].nunique()
    roi_labels          = dict(enumerate(pd.Categorical(df['roi']).categories))
    
    def get_param(df, param_name):
        return (df[df['parameter'] == param_name]
              .sort_values(['participant_id', 'roi'])
              ['median_zscore']
              .values.astype(float))
        
    ufa = get_param(df, 'ufa')
    FA  = get_param(df, 'FA')
    op  = get_param(df, 'op')

    df_ref  = df[df['parameter'] == 'op'].sort_values(['participant_id', 'roi'])
    age_z   = ((df_ref['age'] - df_ref['age'].mean()) / df_ref['age'].std()).values.astype(float)
    sex     = df_ref['sex_numeric'].values.astype(float)
    roi_idx = df_ref['roi_idx_local'].values

    print(f"  {fiber_type} | {n_rois} ROIs, {len(op)} obs")

    with pm.Model() as severity_model:

        # --- Level 2: fiber type hyperprior for ufa---
        mu_beta_ufa  = pm.Normal('mu_beta_ufa', mu=0, sigma=1)
        sig_beta_ufa = pm.HalfNormal('sig_beta_ufa', sigma=0.5)

        # --- Level 1: ROI effects pooled toward fiber type mean ufa ---
        roi_offset_ufa  = pm.Normal('roi_offset_ufa', mu=0, sigma=1, shape=n_rois)
        beta_ufa = pm.Deterministic('beta_ufa',
                                        mu_beta_ufa + roi_offset_ufa * sig_beta_ufa)
        
        # --- Level 2: fiber type hyperprior for FA---
        mu_beta_FA  = pm.Normal('mu_beta_FA', mu=0, sigma=1)
        sig_beta_FA = pm.HalfNormal('sig_beta_FA', sigma=0.5)

        # --- Level 1: ROI effects pooled toward fiber type mean FA ---
        roi_offset_FA  = pm.Normal('roi_offset_FA', mu=0, sigma=1, shape=n_rois)
        beta_FA = pm.Deterministic('beta_FA',
                                        mu_beta_FA + roi_offset_FA * sig_beta_FA)

        # --- Fixed effects ---
        beta_age  = pm.Normal('beta_age',  mu=0, sigma=1)
        beta_sex  = pm.Normal('beta_sex',  mu=0, sigma=1)
        intercept = pm.Normal('intercept', mu=0, sigma=1)

        # --- Residual SD ---
        sigma = pm.HalfNormal('sigma', sigma=1)

        # --- Linear predictor ---
        mu = (intercept +
              beta_ufa[roi_idx] * ufa +
              beta_FA[roi_idx] * FA + 
              beta_age * age_z +
              beta_sex * sex)

        # --- Student-T likelihood instead of Normal ---
        pm.Normal('y', mu=mu, sigma=sigma, observed=op)

        idata = pm.sample(
            5000, tune=5000,
            target_accept=0.95,
            return_inferencedata=True,
            random_seed=8,
            progressbar=True,
        )

        idata.attrs['fiber_type'] = fiber_type
        idata.attrs['roi_labels'] = str(roi_labels)
        idata.attrs['op_mean'] = str(float(op.mean()))
        idata.attrs['op_std']  = str(float(op.std()))   

    return idata


# --- run for all fiber types ---
fiber_types = long_df['tract_type'].unique()

# dict: op_results[fiber_type]
op_results = {}

for fiber_type in fiber_types:
    print(f"\nFitting: {fiber_type}")
    idata = run_op_regression(long_df, fiber_type)
    if idata is not None:
        op_results[fiber_type] = idata


In [ ]:
import arviz as az

for fiber_type, idata in op_results.items():
    if idata is None:
        print(f"  {fiber_type}: no results, skipping")
        continue

    diag = az.summary(
        idata,
        var_names=[
            'mu_beta_ufa', 'beta_ufa', 'sig_beta_ufa',
            'mu_beta_FA',  'beta_FA',  'sig_beta_FA',
            'beta_age', 'beta_sex', 'intercept', 'sigma'
        ],
        hdi_prob=0.95
    )[['r_hat', 'ess_bulk']]

    bad_rhat = diag[diag['r_hat'] > 1.01]
    low_ess  = diag[diag['ess_bulk'] < 400]

    if len(bad_rhat) > 0 or len(low_ess) > 0:
        print(f"WARNING | {fiber_type}:")
        if len(bad_rhat) > 0:
            print(f"  High r_hat:  {bad_rhat.index.tolist()}")
        if len(low_ess) > 0:
            print(f"  Low ESS:     {low_ess.index.tolist()}")
    else:
        n_div = idata.sample_stats['diverging'].values.sum()
        print(f"  ✓ {fiber_type}: converged OK  (divergences: {n_div})")

In [ ]:
def plot_op_contrast(op_results, save_dir=None):
    FIBER_ORDER  = ['association', 'commissural', 'projection']

    n_fibers     = len(FIBER_ORDER)
    n_params     = 2
    group_width  = 0.8
    violin_width = group_width / n_params

    fig, axes = plt.subplots(1, 2, figsize=(n_fibers * 2.5 + 2, 5))

    # --- Left panel: individual mu_betas ---
    ax = axes[0]
    handles = []

    for p_idx, param in enumerate(['ufa', 'FA']):
        color        = PARAM_COLORS.get(param, '#888888')
        positions    = []
        samples_list = []
        medians      = []
        hdis         = []

        for f_idx, fiber_type in enumerate(FIBER_ORDER):
            idata = op_results.get(fiber_type)
            if idata is None:
                continue

            samples = idata.posterior[f'mu_beta_{param}'].values.ravel()
            pos     = f_idx + (p_idx - n_params / 2 + 0.5) * violin_width

            positions.append(pos)
            samples_list.append(samples)
            medians.append(np.median(samples))
            hdis.append(az.hdi(samples, hdi_prob=0.95))

        if not samples_list:
            continue

        parts = ax.violinplot(
            samples_list, positions=positions,
            widths=violin_width * 0.9,
            vert=True, showmedians=False, showextrema=False,
        )
        for pc, hdi in zip(parts['bodies'], hdis):
            contains_zero = hdi[0] < 0 < hdi[1]
            pc.set_facecolor(color)
            pc.set_alpha(0.1 if contains_zero else 0.6)
            pc.set_edgecolor(color)
            pc.set_linewidth(0.8)

        for pos, med, hdi, samp in zip(positions, medians, hdis, samples_list):
            contains_zero = hdi[0] < 0 < hdi[1]
            alpha = 0.1 if contains_zero else 0.6
            ax.plot([pos, pos], [hdi[0], hdi[1]],
                    color=color, linewidth=2.5, alpha=alpha,
                    solid_capstyle='round', zorder=3)
            ax.scatter(pos, med, color=color, alpha=alpha, s=20, zorder=4)
            if hdi[0] > 0 or hdi[1] < 0:
                y_star = np.percentile(samp, 100)
                ax.text(pos, y_star, '*', ha='center', va='bottom',
                        fontsize=20, color=color, zorder=5)

        handles.append(plt.Line2D([0], [0], color=color, linewidth=6,
                                   alpha=0.6, label=param))

    ax.axhline(0, color='gray', linewidth=1.2, linestyle='--', zorder=1)
    ax.set_xticks(np.arange(n_fibers))
    ax.set_xticklabels([f.capitalize() for f in FIBER_ORDER], fontsize=18)
    ax.tick_params(axis='y', labelsize=14)
    ax.set_ylabel(r" $\beta_{param}$ ", fontsize=20)
    ax.set_title("Individual Effects", fontsize=20)
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(handles=handles, frameon=False, fontsize=14, labels=[r'$\mu$FA', 'FA'])

    # --- Right panel: FA - ufa contrast ---
    ax = axes[1]
    color_contrast = "#75717A"
    positions    = []
    samples_list = []
    medians      = []
    hdis         = []

    for f_idx, fiber_type in enumerate(FIBER_ORDER):
        idata = op_results.get(fiber_type)
        if idata is None:
            continue

        samples_ufa = idata.posterior['mu_beta_ufa'].values.ravel()
        samples_FA  = idata.posterior['mu_beta_FA'].values.ravel()
        samples     = samples_FA - samples_ufa

        positions.append(f_idx)
        samples_list.append(samples)
        medians.append(np.median(samples))
        hdis.append(az.hdi(samples, hdi_prob=0.95))

    if samples_list:
        parts = ax.violinplot(
            samples_list, positions=positions,
            widths=0.5,
            vert=True, showmedians=False, showextrema=False,
        )
        for pc, hdi in zip(parts['bodies'], hdis):
            contains_zero = hdi[0] < 0 < hdi[1]
            pc.set_facecolor(color_contrast)
            pc.set_alpha(0.1 if contains_zero else 0.6)
            pc.set_edgecolor(color_contrast)
            pc.set_linewidth(0.8)

        for pos, med, hdi, samp in zip(positions, medians, hdis, samples_list):
            contains_zero = hdi[0] < 0 < hdi[1]
            alpha = 0.1 if contains_zero else 0.6
            ax.plot([pos, pos], [hdi[0], hdi[1]],
                    color='black', linewidth=2.5, alpha=alpha,
                    solid_capstyle='round', zorder=3)
            ax.scatter(pos, med, color='black', alpha=alpha, s=20, zorder=4)
            if hdi[0] > 0 or hdi[1] < 0:
                y_star = np.percentile(samp, 100)
                ax.text(pos, y_star, '*', ha='center', va='bottom',
                        fontsize=20, color='black', zorder=5)

    ax.axhline(0, color='gray', linewidth=1.2, linestyle='--', zorder=1)
    ax.set_xticks(np.arange(n_fibers))
    ax.set_xticklabels([f.capitalize() for f in FIBER_ORDER], fontsize=14)
    ax.tick_params(axis='y', labelsize=14)
    ax.set_ylabel(r"$\Delta$ Effect on OP ($FA - \mathrm{\mu} FA$)", fontsize=20)
    ax.set_title("Contrast", fontsize=20)
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_ylim(0,1.4)

    plt.tight_layout()

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        fname = f"{save_dir}/op_contrast_ufa_FA.png"
        plt.savefig(fname, dpi=600, bbox_inches='tight')
        print(f"  ✓ saved: {fname}")
    else:
        plt.show()
    plt.close()


plot_op_contrast(op_results, save_dir=PLOT_DIR)

In [ ]:
def plot_op_regression_lines(op_results, save_dir=None,
                              param_labels=None):
    FIBER_ORDER = ['association', 'commissural', 'projection']

    default_labels = {
        'ufa': r'$\mathrm{\mu}$FA',
        'FA':  'FA',
    }
    labels = {**default_labels, **(param_labels or {})}

    n_fibers = len(FIBER_ORDER)
    fig, axes = plt.subplots(1, n_fibers,
                              figsize=(n_fibers * 4, 4),
                              sharey=True)

    for f_idx, fiber_type in enumerate(FIBER_ORDER):
        ax    = axes[f_idx]
        idata = op_results.get(fiber_type)

        if idata is None:
            ax.set_visible(False)
            continue

        for param in ['ufa', 'FA']:
            color = PARAM_COLORS.get(param, '#888888')

            mu_beta_samples   = idata.posterior[f'mu_beta_{param}'].values.ravel()
            intercept_samples = idata.posterior['intercept'].values.ravel()

            hdi_beta      = az.hdi(mu_beta_samples, hdi_prob=0.94)
            contains_zero = hdi_beta[0] < 0 < hdi_beta[1]
            alpha         = 0.25 if contains_zero else 0.9
            beta_med      = np.median(mu_beta_samples)

            # x range based on observed predictor values
            x_range = np.linspace(-3, 3, 100)

            mu_lines    = (intercept_samples[:, None] +
                           mu_beta_samples[:, None] * x_range[None, :])
            hdi_band    = az.hdi(mu_lines, hdi_prob=0.94)
            med_line    = np.median(mu_lines, axis=0)

            ax.plot(x_range, med_line,
                    color=color, linewidth=2, alpha=alpha, zorder=4,
                    
                    label=labels.get(param, param))
            ax.fill_between(x_range, hdi_band[:, 0], hdi_band[:, 1],
                            color=color, alpha=0.12, zorder=2)

            # beta annotation
            y_pos = 0.95 if param == 'ufa' else 0.78
            ax.text(0.05, y_pos,
                    f'{labels.get(param, param)}: β={beta_med:.2f}\n'
                    f'[{hdi_beta[0]:.2f}, {hdi_beta[1]:.2f}]',
                    transform=ax.transAxes,
                    ha='left', va='top', fontsize=12,
                    color=color,
                    bbox=dict(boxstyle='round,pad=0.2',
                              facecolor='white',
                              edgecolor=color,
                              alpha=0.7))

            if not contains_zero:
                x_star = x_range[-1]
                y_star = med_line[-1]
                ax.text(x_star + 0.05, y_star, '*',
                        ha='left', va='center', fontsize=20, color=color)

        ax.axhline(0, color='gray', linewidth=0.8, linestyle='--', zorder=1)
        ax.axvline(0, color='gray', linewidth=0.8, linestyle='--', zorder=1)
        ax.set_title(fiber_type.capitalize(), fontsize=20)
        ax.set_xlabel("Predictor (z-scored)", fontsize=16)
        ax.tick_params(axis='y', labelsize=16)
        ax.tick_params(axis='x', labelsize=16)
        if f_idx == 0:
            ax.set_ylabel("Predicted OP (z-scored)", fontsize=20)
        ax.spines[['top', 'right']].set_visible(False)
        ax.legend(frameon=False, fontsize=14, loc='lower right')

        plt.tight_layout()

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        fname = f"{save_dir}/op_regression_lines_ufa_FA.png"
        plt.savefig(fname, dpi=600, bbox_inches='tight')
        print(f"  ✓ saved: {fname}")
    else:
        plt.show()
    plt.close()


plot_op_regression_lines(op_results, save_dir=PLOT_DIR)

In [ ]:
def op_results_to_csv(op_results, out_dir='.', hdi_prob=0.95):
    """
    Extract posterior summaries from op_results[fiber_type]
    and write to CSVs:
      1. op_summary_all.csv          — every variable, every model
      2. op_summary_roi_effects.csv  — ROI-level beta_ufa and beta_FA
      3. op_summary_hyperpriors.csv  — fiber-type hyperpriors (mu/sig for ufa and FA)
      4. op_summary_fixed.csv        — fixed effects (age, sex, intercept, sigma)
    """
    import os, ast
    os.makedirs(out_dir, exist_ok=True)

    lo_col = f"hdi_{round((1 - hdi_prob) / 2 * 100, 1)}%"
    hi_col = f"hdi_{round((1 - (1 - hdi_prob) / 2) * 100, 1)}%"

    all_rows = []

    for fiber_type, idata in op_results.items():
        if idata is None:
            print(f"  {fiber_type}: no results, skipping")
            continue

        roi_labels = ast.literal_eval(idata.attrs.get('roi_labels', '{}'))
        print(f"Extracting: {fiber_type}")

        try:
            summary = az.summary(
                idata,
                var_names=[
                    'mu_beta_ufa', 'sig_beta_ufa', 'beta_ufa',
                    'mu_beta_FA',  'sig_beta_FA',  'beta_FA',
                    'beta_age', 'beta_sex', 'intercept', 'sigma'
                ],
                hdi_prob=hdi_prob
            )
        except Exception as e:
            print(f"  ✗ az.summary failed: {e}")
            continue

        for idx_str, row in summary.iterrows():

            roi_index = None
            roi_label = None
            metric    = None

            if idx_str.startswith('beta_ufa['):
                i         = int(idx_str.split('[')[1].rstrip(']'))
                variable  = 'beta_ufa'
                roi_index = i
                roi_label = roi_labels.get(i)
                metric    = 'ufa'

            elif idx_str.startswith('beta_FA['):
                i         = int(idx_str.split('[')[1].rstrip(']'))
                variable  = 'beta_FA'
                roi_index = i
                roi_label = roi_labels.get(i)
                metric    = 'FA'

            elif idx_str in ('mu_beta_ufa', 'sig_beta_ufa'):
                variable = idx_str
                metric   = 'ufa'

            elif idx_str in ('mu_beta_FA', 'sig_beta_FA'):
                variable = idx_str
                metric   = 'FA'

            else:
                variable = idx_str  # beta_age, beta_sex, intercept, sigma

            all_rows.append({
                'fiber_type':  fiber_type,
                'variable':    variable,
                'metric':      metric,
                'roi_index':   roi_index,
                'roi_label':   roi_label,
                'mean':        row['mean'],
                'sd':          row['sd'],
                lo_col:        row[lo_col],
                hi_col:        row[hi_col],
                'r_hat':       row.get('r_hat',    np.nan),
                'ess_bulk':    row.get('ess_bulk', np.nan),
            })

    df_all = pd.DataFrame(all_rows)

    df_all['hdi_excludes_zero'] = (df_all[lo_col] > 0) | (df_all[hi_col] < 0)

    df_roi = df_all[df_all['variable'].isin(['beta_ufa', 'beta_FA'])]

    df_hyper = df_all[df_all['variable'].isin([
        'mu_beta_ufa', 'sig_beta_ufa', 'mu_beta_FA', 'sig_beta_FA'
    ])]

    df_fixed = df_all[df_all['variable'].isin([
        'beta_age', 'beta_sex', 'intercept', 'sigma'
    ])]

    files = {
        'op_summary_all.csv':          df_all,
        'op_summary_roi_effects.csv':  df_roi,
        'op_summary_hyperpriors.csv':  df_hyper,
        'op_summary_fixed.csv':        df_fixed,
    }

    for fname, df in files.items():
        path = os.path.join(out_dir, fname)
        df.to_csv(path, index=False)
        print(f"Wrote: {path}  ({len(df)} rows)")

    return df_all


# --- Run ---
df_op = op_results_to_csv(op_results, out_dir=MODEL_DIR)

In [ ]:
def op_contrast_to_csv(op_results, out_dir='.', hdi_prob=0.95):
    """
    Extract posterior contrast summaries (mu_beta_FA - mu_beta_ufa)
    from op_results[fiber_type] and write to CSV.
    """
    import os
    os.makedirs(out_dir, exist_ok=True)

    lo_col = f"hdi_{round((1 - hdi_prob) / 2 * 100, 1)}%"
    hi_col = f"hdi_{round((1 - (1 - hdi_prob) / 2) * 100, 1)}%"

    all_rows = []

    for fiber_type, idata in op_results.items():
        if idata is None:
            continue

        print(f"Extracting contrast: {fiber_type}")

        s_FA  = idata.posterior['mu_beta_FA'].values.ravel()
        s_ufa = idata.posterior['mu_beta_ufa'].values.ravel()
        samples = s_FA - s_ufa

        hdi  = az.hdi(samples, hdi_prob=hdi_prob)

        all_rows.append({
            'fiber_type':        fiber_type,
            'contrast':          'FA - ufa',
            'param1':            'FA',
            'param2':            'ufa',
            'mean':              float(np.mean(samples)),
            'median':            float(np.median(samples)),
            'sd':                float(np.std(samples)),
            lo_col:              float(hdi[0]),
            hi_col:              float(hdi[1]),
            'hdi_excludes_zero': (hdi[0] > 0) or (hdi[1] < 0),
        })

    df = pd.DataFrame(all_rows)
    path = os.path.join(out_dir, 'op_contrast_FA_ufa.csv')
    df.to_csv(path, index=False)
    print(f"Wrote: {path}  ({len(df)} rows)")

    return df


# --- Run ---
df_op_contrast = op_contrast_to_csv(op_results, out_dir=MODEL_DIR)

In [ ]:
import pymc as pm

def compare_isotropic(long_df, fiber_type, predictors=['vison', 'MD']):

    df = long_df[
        (long_df['tract_type'] == fiber_type)
    ].copy()

    if df.empty:
        print(f"  No data for {fiber_type}, skipping")
        return None

    df['roi_idx_local'] = pd.Categorical(df['roi']).codes
    n_rois              = df['roi_idx_local'].nunique()
    roi_labels          = dict(enumerate(pd.Categorical(df['roi']).categories))

    def get_param(df, param_name):
        return (df[df['parameter'] == param_name]
              .sort_values(['participant_id', 'roi'])
              ['median_zscore']
              .values.astype(float))

    # extract only the requested predictors
    param_data = {p: get_param(df, p) for p in predictors}

    df_ref  = df[df['parameter'] == predictors[0]].sort_values(['participant_id', 'roi'])
    y       = df_ref['group_id'].values.astype(int)
    age_z   = ((df_ref['age'] - df_ref['age'].mean()) / df_ref['age'].std()).values.astype(float)
    sex     = df_ref['sex_numeric'].values.astype(float)
    roi_idx = df_ref['roi_idx_local'].values

    print(f"  {fiber_type} | predictors={predictors} | {n_rois} ROIs, {len(y)} obs")

    with pm.Model() as iso_model:

        beta = {}
        for p in predictors:
            mu    = pm.Normal(f'mu_beta_{p}',  mu=0, sigma=1)
            sig   = pm.HalfNormal(f'sig_beta_{p}', sigma=0.5)
            offset = pm.Normal(f'roi_offset_{p}', mu=0, sigma=1, shape=n_rois)
            beta[p] = pm.Deterministic(f'beta_{p}', mu + offset * sig)

        # --- Fixed effects ---
        beta_age  = pm.Normal('beta_age',  mu=0, sigma=1)
        beta_sex  = pm.Normal('beta_sex',  mu=0, sigma=1)
        intercept = pm.Normal('intercept', mu=0, sigma=1)

        # --- Logistic predictor ---
        logit_p = (intercept +
                   sum(beta[p][roi_idx] * param_data[p] for p in predictors) +
                   beta_age * age_z +
                   beta_sex * sex)

        pm.Bernoulli('y', logit_p=logit_p, observed=y)

        idata = pm.sample(
            5000, tune=5000,
            target_accept=0.95,
            return_inferencedata=True,
            random_seed=8,
            progressbar=True,
        )

        idata.attrs['fiber_type'] = fiber_type
        idata.attrs['roi_labels'] = str(roi_labels)
        idata.attrs['predictors'] = str(predictors)

    return idata


fiber_types = long_df['tract_type'].unique()

iso_results = {}

for fiber_type in fiber_types:
    iso_results[fiber_type] = {}
    for pred_set in [['vison', 'MD'], ['vison', 'miso']]:
        key = '_'.join(pred_set)
        print(f"\nFitting: {fiber_type} | {pred_set}")
        idata = compare_isotropic(long_df, fiber_type,
                                  predictors=pred_set)
        if idata is not None:
            iso_results[fiber_type][key] = idata

In [ ]:
import arviz as az

for fiber_type, model_dict in iso_results.items():
    for key, idata in model_dict.items():
        if idata is None:
            print(f"  {fiber_type} | {key}: no results, skipping")
            continue

        predictors = eval(idata.attrs.get('predictors', '[]'))
        var_names  = (
            [f'mu_beta_{p}' for p in predictors] +
            [f'beta_{p}'    for p in predictors] +
            [f'sig_beta_{p}' for p in predictors] +
            ['beta_age', 'beta_sex', 'intercept']
        )

        diag = az.summary(
            idata,
            var_names=var_names,
            hdi_prob=0.95
        )[['r_hat', 'ess_bulk']]

        bad_rhat = diag[diag['r_hat'] > 1.01]
        low_ess  = diag[diag['ess_bulk'] < 400]

        if len(bad_rhat) > 0 or len(low_ess) > 0:
            print(f"WARNING | {fiber_type} | {key}:")
            if len(bad_rhat) > 0:
                print(f"  High r_hat:  {bad_rhat.index.tolist()}")
            if len(low_ess) > 0:
                print(f"  Low ESS:     {low_ess.index.tolist()}")
        else:
            n_div = idata.sample_stats['diverging'].values.sum()
            print(f"  ✓ {fiber_type} | {key}: converged OK  (divergences: {n_div})")

In [ ]:
def iso_results_to_csv(iso_results, out_dir='.', hdi_prob=0.95):
    """
    Extract posterior summaries from iso_results[fiber_type][key]
    where key = '{pred1}_{pred2}'
    
    Writes:
      1. iso_summary_all.csv         — every variable, every model
      2. iso_summary_roi_effects.csv — ROI-level beta per predictor
      3. iso_summary_hyperpriors.csv — mu_beta and sig_beta per predictor
      4. iso_summary_fixed.csv       — beta_age, beta_sex, intercept
    """
    import os, ast
    os.makedirs(out_dir, exist_ok=True)

    lo_col = f"hdi_{round((1 - hdi_prob) / 2 * 100, 1)}%"
    hi_col = f"hdi_{round((1 - (1 - hdi_prob) / 2) * 100, 1)}%"

    all_rows = []

    for fiber_type, key_dict in iso_results.items():
        for key, idata in key_dict.items():
            if idata is None:
                print(f"  {fiber_type} | {key}: no results, skipping")
                continue

            roi_labels = ast.literal_eval(idata.attrs.get('roi_labels', '{}'))
            predictors = ast.literal_eval(idata.attrs.get('predictors', '[]'))

            print(f"Extracting: {fiber_type} | {key}")

            var_names = (
                [f'mu_beta_{p}'  for p in predictors] +
                [f'sig_beta_{p}' for p in predictors] +
                [f'beta_{p}'     for p in predictors] +
                ['beta_age', 'beta_sex', 'intercept']
            )

            try:
                summary = az.summary(idata, var_names=var_names, hdi_prob=hdi_prob)
            except Exception as e:
                print(f"  ✗ az.summary failed: {e}")
                continue

            for idx_str, row in summary.iterrows():

                roi_index = None
                roi_label = None
                predictor = None
                variable  = idx_str

                # --- ROI-level beta_{p}[i] ---
                matched_roi = False
                for p in predictors:
                    if idx_str.startswith(f'beta_{p}['):
                        i         = int(idx_str.split('[')[1].rstrip(']'))
                        variable  = f'beta_{p}'
                        roi_index = i
                        roi_label = roi_labels.get(i)
                        predictor = p
                        matched_roi = True
                        break

                if not matched_roi:
                    # --- Hyperpriors mu_beta_{p} / sig_beta_{p} ---
                    for p in predictors:
                        if idx_str in (f'mu_beta_{p}', f'sig_beta_{p}'):
                            predictor = p
                            break

                all_rows.append({
                    'fiber_type':   fiber_type,
                    'predictor_set': '_'.join(predictors),
                    'predictor':    predictor,
                    'variable':     variable,
                    'roi_index':    roi_index,
                    'roi_label':    roi_label,
                    'mean':         row['mean'],
                    'sd':           row['sd'],
                    lo_col:         row[lo_col],
                    hi_col:         row[hi_col],
                    'r_hat':        row.get('r_hat',    np.nan),
                    'ess_bulk':     row.get('ess_bulk', np.nan),
                })

    df_all = pd.DataFrame(all_rows)
    df_all['hdi_excludes_zero'] = (df_all[lo_col] > 0) | (df_all[hi_col] < 0)

    # ROI-level betas
    roi_vars  = [f'beta_{p}' for p in ['vison', 'miso', 'MD']]
    df_roi    = df_all[df_all['variable'].isin(roi_vars)]

    # Hyperpriors
    hyper_vars = (
        [f'mu_beta_{p}'  for p in ['vison', 'miso', 'MD']] +
        [f'sig_beta_{p}' for p in ['vison', 'miso', 'MD']]
    )
    df_hyper = df_all[df_all['variable'].isin(hyper_vars)]

    # Fixed effects
    df_fixed = df_all[df_all['variable'].isin(['beta_age', 'beta_sex', 'intercept'])]

    files = {
        'iso_summary_all.csv':          df_all,
        'iso_summary_roi_effects.csv':  df_roi,
        'iso_summary_hyperpriors.csv':  df_hyper,
        'iso_summary_fixed.csv':        df_fixed,
    }

    for fname, df in files.items():
        path = os.path.join(out_dir, fname)
        df.to_csv(path, index=False)
        print(f"Wrote: {path}  ({len(df)} rows)")

    return df_all


# --- Run ---
df_iso = iso_results_to_csv(iso_results, out_dir=MODEL_DIR)

In [ ]:
def plot_iso_contrast(iso_results, save_dir=None):
    FIBER_ORDER = ['association', 'commissural', 'projection']

    CONTRAST_KEYS = [
        ('vison_MD',   'vison', 'MD'),
        ('vison_miso', 'vison', 'miso'),
    ]

    n_fibers     = len(FIBER_ORDER)
    group_width  = 0.8
    violin_width = group_width / 2

    for model_key, param1, param2 in CONTRAST_KEYS:

        fig, axes = plt.subplots(1, 2, figsize=(n_fibers * 5, 6), gridspec_kw={'wspace': 0.4})

        # -------------------------------------------------------
        # Left panel: individual HDIs for param1 and param2
        # -------------------------------------------------------
        ax = axes[0]
        handles = []

        for p_idx, param in enumerate([param1, param2]):
            color        = PARAM_COLORS.get(param, '#888888')
            positions    = []
            samples_list = []
            medians      = []
            hdis         = []

            for f_idx, fiber_type in enumerate(FIBER_ORDER):
                idata = iso_results.get(fiber_type, {}).get(model_key)
                if idata is None:
                    continue

                samples = idata.posterior[f'mu_beta_{param}'].values.ravel()
                pos     = f_idx + (p_idx - 2 / 2 + 0.5) * violin_width

                positions.append(pos)
                samples_list.append(samples)
                medians.append(np.median(samples))
                hdis.append(az.hdi(samples, hdi_prob=0.95))

            if not samples_list:
                continue

            parts = ax.violinplot(
                samples_list, positions=positions,
                widths=violin_width * 0.9,
                vert=True, showmedians=False, showextrema=False,
            )
            for pc, hdi in zip(parts['bodies'], hdis):
                contains_zero = hdi[0] < 0 < hdi[1]
                pc.set_facecolor(color)
                pc.set_alpha(0.1 if contains_zero else 0.6)
                pc.set_edgecolor(color)
                pc.set_linewidth(0.8)

            for pos, med, hdi, samp in zip(positions, medians, hdis, samples_list):
                contains_zero = hdi[0] < 0 < hdi[1]
                alpha = 0.1 if contains_zero else 0.6
                ax.plot([pos, pos], [hdi[0], hdi[1]],
                        color='black', linewidth=2.5, alpha=alpha,
                        solid_capstyle='round', zorder=3)
                ax.scatter(pos, med, color='black', alpha=alpha,
                           s=20, zorder=4, linewidth=1)
                if hdi[0] > 0 or hdi[1] < 0:
                    y_star = np.percentile(samp, 100)
                    ax.text(pos, y_star, '*', ha='center', va='bottom',
                            fontsize=20, color='black', zorder=5)

            handles.append(plt.Line2D([0], [0], color=color, linewidth=6,
                                       alpha=0.6))

        ax.axhline(0, color='gray', linewidth=1.2, linestyle='--', zorder=1)
        ax.set_xticks(np.arange(n_fibers))
        ax.set_xticklabels([f.capitalize() for f in FIBER_ORDER], fontsize=18)
        ax.tick_params(axis='y', labelsize=18)
        ax.set_ylabel("Log-odds of Concussion", fontsize=22)
        ax.set_title("Individual Effects", fontsize=20)
        ax.spines[['top', 'right']].set_visible(False)
        ax.set_ylim(-1.3,2)

        if param2 == 'miso':
            ax.legend(handles=handles, frameon=False, fontsize=14,
                loc='upper left', bbox_to_anchor=(0, 1), labels=[r'$\hat{V}(D_{\mathrm{iso}})$', r"$D_{iso}$"])
        elif param2 == 'MD':
            ax.legend(handles=handles, frameon=False, fontsize=14,
                loc='upper left', bbox_to_anchor=(0, 1), labels=[r'$\hat{V}(D_{\mathrm{iso}})$', r"MD"])


        # -------------------------------------------------------
        # Right panel: contrast param1 - param2
        # -------------------------------------------------------
        ax = axes[1]
        contrast_color = "#75717A"
        positions    = []
        samples_list = []
        medians      = []
        hdis         = []

        for f_idx, fiber_type in enumerate(FIBER_ORDER):
            idata = iso_results.get(fiber_type, {}).get(model_key)
            if idata is None:
                continue

            s1      = idata.posterior[f'mu_beta_{param1}'].values.ravel()
            s2      = idata.posterior[f'mu_beta_{param2}'].values.ravel()
            samples = s1 - s2

            positions.append(f_idx)
            samples_list.append(samples)
            medians.append(np.median(samples))
            hdis.append(az.hdi(samples, hdi_prob=0.95))

        if samples_list:
            parts = ax.violinplot(
                samples_list, positions=positions,
                widths=0.5,
                vert=True, showmedians=False, showextrema=False,
            )
            for pc, hdi in zip(parts['bodies'], hdis):
                contains_zero = hdi[0] < 0 < hdi[1]
                pc.set_facecolor(contrast_color)
                pc.set_alpha(0.1 if contains_zero else 0.6)
                pc.set_edgecolor(contrast_color)
                pc.set_linewidth(0.8)

            for pos, med, hdi, samp in zip(positions, medians, hdis, samples_list):
                contains_zero = hdi[0] < 0 < hdi[1]
                alpha = 0.1 if contains_zero else 0.6
                ax.plot([pos, pos], [hdi[0], hdi[1]],
                        color='black', linewidth=2.5, alpha=alpha,
                        solid_capstyle='round', zorder=3)
                ax.scatter(pos, med, color='black', alpha=alpha,
                           s=20, zorder=4, linewidth=1)
                if hdi[0] > 0 or hdi[1] < 0:
                    y_star = np.percentile(samp, 100)
                    ax.text(pos, y_star, '*', ha='center', va='bottom',
                            fontsize=20, color='black', zorder=5)

        ax.axhline(0, color='gray', linewidth=1.2, linestyle='--', zorder=1)
        ax.set_xticks(np.arange(n_fibers))
        ax.set_xticklabels([f.capitalize() for f in FIBER_ORDER], fontsize=18)
        ax.tick_params(axis='y', labelsize=18)
        ax.set_ylim(-1.3,2)

        if param2 == 'MD':
            ax.set_ylabel(r"$\Delta$ Log-odds of Con. ($\hat{V}(D_{\mathrm{iso}}) - MD$)", fontsize=22)
        elif param2 == 'miso':
            ax.set_ylabel(r"$\Delta$ Log-odds of Con. ($\hat{V}(D_{\mathrm{iso}}) - D_{iso}$)", fontsize=22)

        ax.set_title("Contrast", fontsize=20)
        ax.spines[['top', 'right']].set_visible(False)

        plt.tight_layout()

        if save_dir:
            os.makedirs(save_dir, exist_ok=True)
            fname = f"{save_dir}/iso_contrast_{param1}_{param2}.png"
            plt.savefig(fname, dpi=600, bbox_inches='tight')
            print(f"  ✓ saved: {fname}")
        else:
            plt.show()
        plt.close()


plot_iso_contrast(iso_results, save_dir=PLOT_DIR)

In [59]:
def iso_contrast_to_csv(iso_results, out_dir='.', hdi_prob=0.95):
    """
    Extract posterior contrast summaries (mu_beta_param1 - mu_beta_param2)
    from iso_results and write to CSV.
    """
    import os, ast
    os.makedirs(out_dir, exist_ok=True)

    FIBER_ORDER = ['association', 'commissural', 'projection']

    CONTRAST_KEYS = [
        ('vison', 'MD'),
        ('vison', 'miso'),
    ]

    lo_col = f"hdi_{round((1 - hdi_prob) / 2 * 100, 1)}%"
    hi_col = f"hdi_{round((1 - (1 - hdi_prob) / 2) * 100, 1)}%"

    all_rows = []

    for param1, param2 in CONTRAST_KEYS:
        model_key = f'{param1}_{param2}'

        for fiber_type in FIBER_ORDER:
            idata = iso_results.get(fiber_type, {}).get(model_key)
            if idata is None:
                continue

            s1      = idata.posterior[f'mu_beta_{param1}'].values.ravel()
            s2      = idata.posterior[f'mu_beta_{param2}'].values.ravel()
            samples = s1 - s2

            hdi     = az.hdi(samples, hdi_prob=hdi_prob)
            med     = float(np.median(samples))
            mean    = float(np.mean(samples))
            sd      = float(np.std(samples))

            all_rows.append({
                'fiber_type':        fiber_type,
                'contrast':          f'{param1} - {param2}',
                'param1':            param1,
                'param2':            param2,
                'mean':              mean,
                'median':            med,
                'sd':                sd,
                lo_col:              float(hdi[0]),
                hi_col:              float(hdi[1]),
                'hdi_excludes_zero': (hdi[0] > 0) or (hdi[1] < 0),
            })

    df = pd.DataFrame(all_rows)

    path = os.path.join(out_dir, 'iso_contrast_summary.csv')
    df.to_csv(path, index=False)
    print(f"Wrote: {path}  ({len(df)} rows)")

    return df


# --- Run ---
df_contrast = iso_contrast_to_csv(iso_results, out_dir=MODEL_DIR)

Wrote: ./model_outputs/iso_contrast_summary.csv  (18 rows)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_curve, auc
import numpy as np
import pandas as pd

COVARIATES = ['age', 'sex_numeric', 'op']
LABEL_COL  = 'group_id'
N_SPLITS   = 4
N_REPEATS  = 10
SEED       = 42


def crossval_auc(df, param, roi, covariates=['age', 'sex_numeric'],
                 label=LABEL_COL, n_splits=N_SPLITS, n_repeats=N_REPEATS,
                 control_op=False):
    """
    Run repeated stratified k-fold logistic regression for one param x roi.
    Optionally controls for OP as a covariate.
    """
    # get the parameter of interest
    subset = df[
        (df['parameter']  == param)     &
        (df['roi']        == roi)
    ][['participant_id', 'median_zscore'] + covariates + [label]].dropna()

    if subset.empty or subset[label].nunique() < 2:
        return np.nan, None, None

    # --- merge OP as covariate if requested ---
    if control_op and param != 'op':
        op_vals = df[
            (df['parameter'] == 'op') &
            (df['roi']       == roi)
        ][['participant_id', 'median_zscore']].rename(
            columns={'median_zscore': 'op_zscore'})

        subset = subset.merge(op_vals, on='participant_id', how='left')
        feature_cols = ['median_zscore'] + covariates + ['op_zscore']
    else:
        feature_cols = ['median_zscore'] + covariates

    subset = subset.dropna(subset=feature_cols)

    if subset.empty or subset[label].nunique() < 2:
        return np.nan, None, None

    X = subset[feature_cols].astype(float).to_numpy()
    y = subset[label].astype(int).to_numpy()

    skf = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats,
                                   random_state=SEED)
    all_preds = []
    all_trues = []

    for tr, te in skf.split(X, y):
        if len(np.unique(y[tr])) < 2:
            continue
        scaler  = StandardScaler()
        X_train = scaler.fit_transform(X[tr])
        X_test  = scaler.transform(X[te])

        model = LogisticRegression(max_iter=300, solver='lbfgs')
        model.fit(X_train, y[tr])
        probs = model.predict_proba(X_test)[:, 1]

        all_preds.extend(probs)
        all_trues.extend(y[te])

    if len(np.unique(all_trues)) < 2:
        return np.nan, None, None

    fpr, tpr, _ = roc_curve(all_trues, all_preds)
    return auc(fpr, tpr), np.array(all_preds), np.array(all_trues)


# --- run for all parameter x roi combinations ---
params = long_df['parameter'].unique()
rois   = long_df['roi'].unique()

auc_records = []

for param in params:
    for roi in rois:
        tract_type = long_df[long_df['roi'] == roi]['tract_type'].iloc[0]
        print(f"  {param} | {roi}...")

        auc_val, preds, trues = crossval_auc(
            long_df, param, roi,
            control_op=False)

        auc_records.append({
            'parameter':  param,
            'roi':        roi,
            'tract_type': tract_type,
            'auc':        auc_val,
        })

auc_df = pd.DataFrame(auc_records).sort_values(
            ['parameter', 'tract_type', 'roi']).reset_index(drop=True)

print(auc_df)

In [ ]:
from scipy import stats

def plot_auc_violins_comparison(auc_df, save_dir=None):
    FIBER_ORDER = ['association', 'commissural', 'projection']

    # two panels: anisotropic and isotropic DTD vs DTI
    PANELS = [
        ('Anisotropic DTD vs DTI', ['ufa', 'FA', 'MD'],   [('ufa', 'FA'), ('ufa', 'MD')], {'ufa': r'$\mu$FA', 'FA': 'FA', 'MD': 'MD'}),
        ('Isotropic DTD vs DTI',   ['vison', 'FA', 'MD'], [('vison', 'FA'), ('vison', 'MD')], {'vison': r'$\hat{V}(D_{\mathrm{iso}})$', 'FA': 'FA', 'MD': 'MD'}),
    ]

    n_fibers = len(FIBER_ORDER)
    fig, axes = plt.subplots(1, 2, figsize=(n_fibers * 5, 6))

    for ax, (panel_title, param_order, comparisons, panel_labels) in zip(axes, PANELS):

        n_params     = len(param_order)
        group_width  = 0.8
        violin_width = group_width / n_params
        handles      = []
        param_fiber_data = {}

        for p_idx, param in enumerate(param_order):
            color     = PARAM_COLORS.get(param, '#888888')
            positions    = []
            samples_list = []
            medians      = []
            hdis         = []

            for f_idx, fiber_type in enumerate(FIBER_ORDER):
                subset = auc_df[
                    (auc_df['parameter']  == param)      &
                    (auc_df['tract_type'] == fiber_type)
                ]['auc'].dropna().values

                if len(subset) == 0:
                    continue

                pos = f_idx + (p_idx - n_params / 2 + 0.5) * violin_width
                positions.append(pos)
                samples_list.append(subset)
                medians.append(np.median(subset))
                hdis.append(az.hdi(subset, hdi_prob=0.95) if len(subset) > 1 else None)
                param_fiber_data[(param, fiber_type)] = (pos, subset)

            if not samples_list:
                continue

            parts = ax.violinplot(
                samples_list, positions=positions,
                widths=violin_width * 0.9,
                vert=True, showmedians=False, showextrema=False,
            )
            for pc in parts['bodies']:
                pc.set_facecolor(color)
                pc.set_alpha(0.4)
                pc.set_edgecolor(color)
                pc.set_linewidth(0.8)

            for pos, med, vals in zip(positions, medians, samples_list):
                # individual ROI points with jitter
                jitter = np.random.uniform(-violin_width * 0.15,
                                            violin_width * 0.15,
                                            size=len(vals))
                ax.scatter(pos + jitter, vals,
                           color=color, alpha=0.4, s=15, zorder=4)

                # HDI line
                if len(vals) > 1:
                    hdi = az.hdi(vals, hdi_prob=0.95)
                    ax.plot([pos, pos], [hdi[0], hdi[1]],
                            color='black', linewidth=2.5, alpha=0.8,
                            solid_capstyle='round', zorder=3)

                # median diamond
                ax.scatter(pos, med, color='black', alpha=1.0,
                           s=50, zorder=5, marker='D',
                           edgecolors='white', linewidth=0.5)

            handles.append(plt.Line2D([0], [0], color=color, linewidth=6,
                                       alpha=0.6, label=panel_labels.get(param, param)))

        # --- t-tests ---
        y_max        = auc_df[
            auc_df['parameter'].isin(param_order)
        ]['auc'].max()
        bracket_base = y_max + 0.03
        bracket_step = 0.06

        print(f"\n=== {panel_title} ===")

        for c_idx, (p1, p2) in enumerate(comparisons):
            print(f"\n  {p1} vs {p2}:")
            for fiber_type in FIBER_ORDER:
                if (p1, fiber_type) not in param_fiber_data:
                    continue
                if (p2, fiber_type) not in param_fiber_data:
                    continue

                pos1, vals1 = param_fiber_data[(p1, fiber_type)]
                pos2, vals2 = param_fiber_data[(p2, fiber_type)]

                t, p = stats.ttest_rel(vals1, vals2)
                sig  = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'

                print(f"    {fiber_type}: t={t:.3f}, p={p:.3f} {sig}")

                if p < 0.05:
                    y_bracket = bracket_base + c_idx * bracket_step
                    mid       = (pos1 + pos2) / 2
                    ax.plot([pos1, pos1, pos2, pos2],
                            [y_bracket - 0.01, y_bracket,
                             y_bracket, y_bracket - 0.01],
                            color='black', linewidth=1, zorder=6)
                    ax.text(mid, y_bracket + 0.005, sig,
                            ha='center', va='bottom', fontsize=20,
                            color='black', zorder=7)

        ax.axhline(0.7, color='gray', linewidth=1.2, linestyle='--', zorder=1)
        ax.set_xticks(np.arange(n_fibers))
        ax.set_xticklabels([f.capitalize() for f in FIBER_ORDER], fontsize=18)
        ax.tick_params(axis='y', labelsize=14)
        ax.set_ylabel("AUC", fontsize=20)
        ax.set_ylim(0.3, bracket_base + len(comparisons) * bracket_step + 0.05)
        #ax.set_title(panel_title, fontsize=14)
        ax.spines[['top', 'right']].set_visible(False)
        ax.legend(handles=handles, frameon=False, fontsize=14,
                  loc='upper left', bbox_to_anchor=(0, 1))
        ax.set_ylim(0.5,1)

    plt.tight_layout()

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        fname = f"{save_dir}/auc_dtd_vs_dti.png"
        plt.savefig(fname, dpi=600, bbox_inches='tight')
        print(f"  ✓ saved: {fname}")
    else:
        plt.show()
    plt.close()


plot_auc_violins_comparison(auc_df, save_dir=PLOT_DIR)

In [ ]:
def auc_comparison_to_csv(auc_df, out_dir='.'):
    """
    Extract AUC summaries and paired t-test results from auc_df and write to CSVs:
      1. auc_summary.csv   — median, HDI, and descriptives per param/fiber
      2. auc_ttests.csv    — paired t-test results for each comparison
    """
    import os
    from scipy import stats
    os.makedirs(out_dir, exist_ok=True)

    FIBER_ORDER = ['association', 'commissural', 'projection']

    PANELS = [
        ('anisotropic', ['ufa', 'FA', 'MD'],   [('ufa', 'FA'), ('ufa', 'MD')]),
        ('isotropic',   ['vison', 'FA', 'MD'], [('vison', 'FA'), ('vison', 'MD')]),
    ]

    # ------------------------------------------------------------------
    # 1. AUC summary rows
    # ------------------------------------------------------------------
    summary_rows = []

    for panel_name, param_order, _ in PANELS:
        for param in param_order:
            for fiber_type in FIBER_ORDER:

                subset = auc_df[
                    (auc_df['parameter']  == param)      &
                    (auc_df['tract_type'] == fiber_type)
                ]['auc'].dropna().values

                if len(subset) == 0:
                    continue

                hdi = az.hdi(subset, hdi_prob=0.95) if len(subset) > 1 else [np.nan, np.nan]

                summary_rows.append({
                    'panel':       panel_name,
                    'parameter':   param,
                    'fiber_type':  fiber_type,
                    'n':           len(subset),
                    'mean':        float(np.mean(subset)),
                    'median':      float(np.median(subset)),
                    'sd':          float(np.std(subset)),
                    'hdi_2.5%':    float(hdi[0]),
                    'hdi_97.5%':   float(hdi[1]),
                    'above_chance': float(hdi[0]) > 0.5,
                })

    df_summary = pd.DataFrame(summary_rows)
    path_summary = os.path.join(out_dir, 'auc_summary.csv')
    df_summary.to_csv(path_summary, index=False)
    print(f"Wrote: {path_summary}  ({len(df_summary)} rows)")

    # ------------------------------------------------------------------
    # 2. Paired t-test rows
    # ------------------------------------------------------------------
    ttest_rows = []

    for panel_name, param_order, comparisons in PANELS:
        for p1, p2 in comparisons:
            for fiber_type in FIBER_ORDER:

                vals1 = auc_df[
                    (auc_df['parameter']  == p1)         &
                    (auc_df['tract_type'] == fiber_type)
                ]['auc'].dropna().values

                vals2 = auc_df[
                    (auc_df['parameter']  == p2)         &
                    (auc_df['tract_type'] == fiber_type)
                ]['auc'].dropna().values

                if len(vals1) == 0 or len(vals2) == 0:
                    continue

                t, p = stats.ttest_rel(vals1, vals2)
                sig  = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'

                ttest_rows.append({
                    'panel':        panel_name,
                    'fiber_type':   fiber_type,
                    'comparison':   f'{p1} vs {p2}',
                    'param1':       p1,
                    'param2':       p2,
                    'mean_auc_p1':  float(np.mean(vals1)),
                    'mean_auc_p2':  float(np.mean(vals2)),
                    'mean_diff':    float(np.mean(vals1) - np.mean(vals2)),
                    't_stat':       float(t),
                    'p_value':      float(p),
                    'sig':          sig,
                })

    df_ttests = pd.DataFrame(ttest_rows)
    path_ttests = os.path.join(out_dir, 'auc_ttests.csv')
    df_ttests.to_csv(path_ttests, index=False)
    print(f"Wrote: {path_ttests}  ({len(df_ttests)} rows)")

    roi_rows = []

    for panel_name, param_order, _ in PANELS:
        for param in param_order:
            for fiber_type in FIBER_ORDER:

                subset = auc_df[
                    (auc_df['parameter']  == param)      &
                    (auc_df['tract_type'] == fiber_type)
                ][['roi', 'auc']].dropna()

                if subset.empty:
                    continue

                for _, row in subset.iterrows():
                    roi_rows.append({
                        'panel':      panel_name,
                        'parameter':  param,
                        'fiber_type': fiber_type,
                        'roi':        row['roi'],
                        'auc':        float(row['auc']),
                        'above_chance': float(row['auc']) > 0.5,
                    })

    df_roi = pd.DataFrame(roi_rows)
    path_roi = os.path.join(out_dir, 'auc_roi.csv')
    df_roi.to_csv(path_roi, index=False)
    print(f"Wrote: {path_roi}  ({len(df_roi)} rows)")

    return df_summary, df_ttests, roi_rows


# --- Run ---
df_auc_summary, df_auc_ttests, df_auc_roi = auc_comparison_to_csv(auc_df, out_dir=MODEL_DIR)